# Day 1: Tensors, Data Loading & Normalization
## Defense Prep - Coding Exercises (Offline Model Track)

**Date**: March 23, 2026
**Focus**: The building blocks you need BEFORE you can build a neural network

**Why start here?** Your offline model takes raw CSV data `(x,y,z,t,Vx,Vy,P,TKE)`, loads it, normalizes it, and feeds it to a network. Today you master every step BEFORE the network.

**Rules**:
- Write your solution in the empty code cells
- Run the test cells (marked `=== TEST ===`) to verify
- Do NOT look at your thesis notebooks until all exercises are done
- Think about WHY each step exists, not just HOW

---

In [2]:
import torch
import torch.nn as nn
import numpy as np
import os
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

PyTorch version: 2.2.2
CUDA available: False
Using device: cpu


---
## Part A: Tensor Fundamentals
---

### Exercise 1: Tensor Creation & Data Types

Neural networks operate on tensors. Your dataset has 7.9 million rows of float values. Understanding tensor creation and dtypes is essential.

**Task**: Create the following tensors:
- `a`: A tensor `[1, 2, 3, 4]` with dtype `float32` (default for neural networks)
- `b`: A tensor of shape `(3, 4)` filled with zeros, dtype `float32`
- `c`: A tensor of shape `(100, 8)` filled with random values from uniform [0, 1) -- this simulates 100 data rows with 8 columns like your CSV
- `d`: An integer tensor `[0, 1, 2, ..., 299]` -- this simulates 300 timestep indices

In [16]:
# YOUR CODE HERE
a = torch.tensor([1,2,3,4], dtype= torch.float32)
b = torch.zeros([3,4], dtype = torch.float32)
c = torch.rand(100,8)
d = torch.arange(0,300)

In [17]:
# === TEST 1 ===
assert a.shape == (4,) and a.dtype == torch.float32, 'a: wrong shape or dtype'
assert torch.equal(a, torch.tensor([1.0, 2.0, 3.0, 4.0])), 'a: wrong values'

assert b.shape == (3, 4) and b.dtype == torch.float32, 'b: wrong shape or dtype'
assert torch.all(b == 0), 'b: should be all zeros'

assert c.shape == (100, 8) and c.dtype == torch.float32, 'c: wrong shape or dtype'
assert c.min() >= 0 and c.max() < 1, 'c: values should be in [0, 1)'

assert d.shape == (300,), 'd: wrong shape'
assert d[0] == 0 and d[-1] == 299, 'd: wrong values'
assert d.dtype in (torch.int64, torch.int32), 'd: should be integer type'

print('Test 1 passed!')

Test 1 passed!


### Exercise 2: Slicing -- Splitting Inputs from Targets

Your CSV has 8 columns: `[x, y, z, t, Vx, Vy, Pressure, TKE]`
- First 4 columns = **inputs** (spatial coordinates + time)
- Last 4 columns = **targets** (field variables the network predicts)

**Task**: Given `raw_data` of shape `(500, 8)`, extract:
- `inputs`: columns 0-3, shape `(500, 4)`
- `targets`: columns 4-7, shape `(500, 4)`
- `spatial_coords`: columns 0-2 only (x, y, z), shape `(500, 3)`
- `time_col`: column 3 only (t), shape `(500,)` -- 1D, not 2D!
- `first_10`: first 10 rows of raw_data, shape `(10, 8)`

In [22]:
torch.manual_seed(42)
raw_data = torch.rand(500, 8)  # simulates 500 rows of your CSV

# YOUR CODE HERE
inputs = raw_data[:, 0:4]
targets = raw_data[:, 4:8]
spatial_coords = raw_data[:, 0:3]
time_col = raw_data[:, 3]
first_10 = raw_data[0:10, :]

In [23]:
# === TEST 2 ===
assert inputs.shape == (500, 4), 'inputs shape wrong: {}'.format(inputs.shape)
assert targets.shape == (500, 4), 'targets shape wrong: {}'.format(targets.shape)
assert spatial_coords.shape == (500, 3), 'spatial_coords shape wrong'
assert time_col.shape == (500,), 'time_col should be 1D with shape (500,), got {}'.format(time_col.shape)
assert first_10.shape == (10, 8), 'first_10 shape wrong'

# Verify slicing correctness
assert torch.equal(inputs, raw_data[:, :4]), 'inputs should be first 4 columns'
assert torch.equal(targets, raw_data[:, 4:]), 'targets should be last 4 columns'
assert torch.equal(spatial_coords, raw_data[:, :3]), 'spatial_coords wrong'
assert torch.equal(time_col, raw_data[:, 3]), 'time_col wrong'
assert torch.equal(first_10, raw_data[:10]), 'first_10 wrong'

print('Test 2 passed!')

Test 2 passed!


### Exercise 3: Reshaping & Dimensions

Different parts of your pipeline need data in different shapes:
- Training: `(N, 4)` -- flat batch of 4D vectors
- SSIM metric: `(1, 1, N, 4)` -- torchmetrics requires image-like format
- Conv2D AE: `(batch, channels, H, W)` -- grid images

**Task**: Given `pred` of shape `(1000, 4)` (1000 predictions, 4 field variables):
- `ssim_shape`: Reshape to `(1, 1, 1000, 4)` for SSIM computation. Use `.view()` then `.permute()`
  - Step 1: `pred.view(-1, 1, 1, 4)` gives `(1000, 1, 1, 4)`
  - Step 2: `.permute(1, 2, 0, 3)` gives `(1, 1, 1000, 4)`
- `flat`: Flatten to 1D tensor of shape `(4000,)`
- `batched`: Reshape to `(10, 100, 4)` -- as if 10 batches of 100 samples

In [36]:
torch.manual_seed(0)
pred = torch.randn(1000, 4)

# YOUR CODE HERE
ssim_shape = pred.view(-1, 1, 1, 4)
ssim_shape = ssim_shape.permute(1,2,0,3)
flat = pred.reshape(4000, )
batched = pred.reshape(10, 100, 4)

In [37]:
# === TEST 3 ===
assert ssim_shape.shape == (1, 1, 1000, 4), 'ssim_shape wrong: {}'.format(ssim_shape.shape)
# Verify the values are correctly placed
assert ssim_shape[0, 0, 0, 0] == pred[0, 0], 'ssim_shape values misaligned'
assert ssim_shape[0, 0, 999, 3] == pred[999, 3], 'ssim_shape last value wrong'

assert flat.shape == (4000,), 'flat shape wrong: {}'.format(flat.shape)
assert flat.numel() == pred.numel(), 'flat should have same number of elements'

assert batched.shape == (10, 100, 4), 'batched shape wrong: {}'.format(batched.shape)

print('Test 3 passed!')

Test 3 passed!


### Exercise 4: NumPy <-> Tensor Conversion

Your pipeline loads data as NumPy arrays (from CSV via pyarrow/pandas) and converts to PyTorch tensors for training. After prediction, you convert back to NumPy for visualization.

**Task**:
- `tensor_from_np`: Convert `np_array` to a PyTorch float32 tensor
- `np_from_tensor`: Convert `some_tensor` back to a NumPy array
- `gpu_to_np`: Move `gpu_tensor` from GPU (or CPU) to NumPy. Remember: you must `.cpu()` before `.numpy()` if on GPU

**Thesis connection**: Your `SpatioTemporalDataset.__getitem__` returns `torch.from_numpy()` tensors. Your visualization code calls `.cpu().numpy()` on predictions.

In [31]:
np_array = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], dtype=np.float64)
some_tensor = torch.tensor([[10.0, 20.0], [30.0, 40.0]])
gpu_tensor = torch.randn(5, 3).to(device)  # on GPU if available, else CPU

# YOUR CODE HERE
tensor_from_np = torch.tensor(np_array, dtype= torch.float32)  # Should be float32, NOT float64
np_from_tensor = some_tensor.numpy()  # Should be a numpy array
gpu_to_np = gpu_tensor.cpu().numpy()    # Should be a numpy array regardless of device

In [32]:
# === TEST 4 ===
assert isinstance(tensor_from_np, torch.Tensor), 'tensor_from_np must be a Tensor'
assert tensor_from_np.dtype == torch.float32, 'Must be float32, got {}'.format(tensor_from_np.dtype)
assert tensor_from_np.shape == (3, 2), 'Shape wrong'

assert isinstance(np_from_tensor, np.ndarray), 'np_from_tensor must be numpy array'
assert np_from_tensor.shape == (2, 2), 'Shape wrong'
assert np.allclose(np_from_tensor, np.array([[10, 20], [30, 40]])), 'Values wrong'

assert isinstance(gpu_to_np, np.ndarray), 'gpu_to_np must be numpy array'
assert gpu_to_np.shape == (5, 3), 'Shape wrong'

print('Test 4 passed!')

Test 4 passed!


### Exercise 5: Device Management (CPU vs GPU)

Your training scripts auto-detect GPU and move both model and data to the same device. Mismatched devices cause runtime errors.

**Task**:
- `t1`: Create a tensor `[1.0, 2.0, 3.0]` on `device` (GPU if available)
- `t2`: Create a tensor on CPU, then move it to `device`
- `same_device`: Check if t1 and t2 are on the same device (should be True)
- `t3_cpu`: Move `t1` back to CPU

In [34]:
# YOUR CODE HERE
t1 = torch.tensor([1, 2, 3], dtype = torch.float32, device = device)
t2_cpu = torch.tensor([4.0, 5.0, 6.0])  # starts on CPU
t2 = t2_cpu.to(device)  # move t2_cpu to device
same_device = (t1.device == t2.device)  # boolean: are t1 and t2 on the same device?
t3_cpu = t1.cpu()  # t1 moved back to CPU

In [35]:
# === TEST 5 ===
assert t1.device == device or str(t1.device).startswith(str(device)), 't1 not on correct device'
assert t2.device == t1.device, 't2 should be on same device as t1'
assert same_device == True, 'same_device should be True'
assert t3_cpu.device == torch.device('cpu'), 't3_cpu should be on CPU'

print('Test 5 passed!')

Test 5 passed!


---
## Part B: Loading Data (Your CSV Format)
---

### Exercise 6: Understanding Your Data Structure

Your real CSV (`ML_test_loader_original_data.csv`) has:
- 7,919,100 rows, 8 columns, NO header row
- Columns: `x, y, z, t, Vx, Vy, Pressure, TKE`
- 300 unique timesteps, ~26,397 spatial points per timestep

We will create a **synthetic mini-dataset** that matches this exact structure so you can practice without loading 794 MB.

**Task**: Create `synthetic_data` as a NumPy array with:
- 1000 rows, 8 columns
- Column 0 (x): uniform random in [-0.5, 2.5]
- Column 1 (y): uniform random in [-0.5, 0.5]
- Column 2 (z): all zeros (2D simulation slice)
- Column 3 (t): repeat 10 unique timesteps evenly spaced in [0, 0.396], each appearing 100 times
- Column 4 (Vx): random in [-1, 3]
- Column 5 (Vy): random in [-1, 1]
- Column 6 (Pressure): random in [95000, 105000]
- Column 7 (TKE): random in [0, 50]

The ranges mimic realistic physical values before normalization.

In [3]:
np.random.seed(42)
N = 1000
N_POINTS_PER_T = 100
N_TIMESTEPS = 10

# YOUR CODE HERE
x = np.random.rand(N) * 3.0 - 0.5   # shape (1000,), uniform in [-0.5, 2.5]
y = np.random.rand(N) * 1.0 - 0.5# shape (1000,), uniform in [-0.5, 0.5]
z = np.zeros(N)       # shape (1000,), all zeros
t = np.repeat(np.linspace(0, 0.396, N_TIMESTEPS), N_POINTS_PER_T) # shape (1000,), 10 unique values each repeated 100 times(Hint: np.repeat(np.linspace(0, 0.396, 10), 100))
Vx = np.random.rand(N) * 4.0 - 1.0      # shape (1000,), uniform in [-1, 3]
Vy = np.random.rand(N)*2.0-1.0      # shape (1000,), uniform in [-1, 1]
Pressure = np.random.rand(N)*10000.0+95000.0 # shape (1000,), uniform in [95000, 105000]
TKE = np.random.rand(N)*50.0     # shape (1000,), uniform in [0, 50]

# Stack into (1000, 8) array
synthetic_data = np.column_stack([x,y,z,t,Vx, Vy, Pressure, TKE])  # Use np.column_stack or np.stack with axis=1

In [4]:
# === TEST 6 ===
assert synthetic_data.shape == (1000, 8), 'Shape should be (1000, 8), got {}'.format(synthetic_data.shape)
assert synthetic_data.dtype == np.float64, 'dtype should be float64'

# Check column ranges
assert synthetic_data[:, 0].min() >= -0.5 and synthetic_data[:, 0].max() <= 2.5, 'x range wrong'
assert synthetic_data[:, 1].min() >= -0.5 and synthetic_data[:, 1].max() <= 0.5, 'y range wrong'
assert np.all(synthetic_data[:, 2] == 0), 'z should be all zeros'

# Check timestep structure
unique_times = np.unique(synthetic_data[:, 3])
assert len(unique_times) == 10, 'Should have 10 unique timesteps, got {}'.format(len(unique_times))
for t_val in unique_times:
    count = np.sum(synthetic_data[:, 3] == t_val)
    assert count == 100, 'Each timestep should have 100 points, got {}'.format(count)

assert synthetic_data[:, 6].min() >= 95000, 'Pressure range wrong'
assert synthetic_data[:, 7].min() >= 0 and synthetic_data[:, 7].max() <= 50, 'TKE range wrong'

print('Test 6 passed!')
print('Data shape:', synthetic_data.shape)
print('Unique timesteps:', len(unique_times))
print('Points per timestep:', N_POINTS_PER_T)
print('\nColumn ranges (min -> max):')
col_names = ['x', 'y', 'z', 't', 'Vx', 'Vy', 'Pressure', 'TKE']
for i, name in enumerate(col_names):
    print('  {}: {:.2f} -> {:.2f}'.format(name, synthetic_data[:, i].min(), synthetic_data[:, i].max()))

Test 6 passed!
Data shape: (1000, 8)
Unique timesteps: 10
Points per timestep: 100

Column ranges (min -> max):
  x: -0.49 -> 2.50
  y: -0.50 -> 0.50
  z: 0.00 -> 0.00
  t: 0.00 -> 0.40
  Vx: -1.00 -> 2.99
  Vy: -1.00 -> 1.00
  Pressure: 95000.31 -> 104977.49
  TKE: 0.31 -> 49.97


### Exercise 7: Save and Load CSV (Matching Your Format)

Your real CSV has NO header. You must assign column names manually when loading.

**Task**:
1. Save `synthetic_data` to `../defense_prep/mini_dataset.csv` with NO header
2. Load it back using `pyarrow.csv` with manual column names (exactly like your thesis code)
3. Verify the loaded data matches the original

**Thesis connection**: This is the exact loading pattern from your `SpatioTemporalDataset` class.

In [6]:
import pyarrow.csv as pv

csv_path = '../defense_prep/mini_dataset.csv'

# Step 1: Save synthetic_data to CSV with NO header
# YOUR CODE HERE (hint: np.savetxt or pandas with header=False)
np.savetxt(csv_path, synthetic_data, delimiter=',')
# Step 2: Load it back using pyarrow with manual column names
# YOUR CODE HERE
column_names = ['x', 'y', 'z', 't', 'Vx', 'Vy', 'Pressure', 'TKE']
read_options = pv.ReadOptions(column_names = column_names)  # pv.ReadOptions(column_names=column_names)
table = pv.read_csv(csv_path, read_options=read_options)         # pv.read_csv(csv_path, read_options=read_options)
loaded_data = table.to_pandas().values  # convert to numpy: table.to_pandas().values

In [7]:
# === TEST 7 ===
assert os.path.exists(csv_path), 'CSV file not saved'
assert loaded_data.shape == (1000, 8), 'Loaded data shape wrong: {}'.format(loaded_data.shape)
assert np.allclose(loaded_data, synthetic_data, atol=1e-6), 'Loaded data does not match original'

# Check that column names were assigned
df = table.to_pandas()
assert list(df.columns) == column_names, 'Column names not set correctly: {}'.format(list(df.columns))

print('Test 7 passed!')
print('File size: {:.1f} KB'.format(os.path.getsize(csv_path) / 1024))
print('Columns:', list(df.columns))

Test 7 passed!
File size: 196.7 KB
Columns: ['x', 'y', 'z', 't', 'Vx', 'Vy', 'Pressure', 'TKE']


### Exercise 8: Split into Inputs and Targets (NumPy)

After loading the CSV, the first thing your `SpatioTemporalDataset` does is split the 8-column array into:
- **Inputs**: columns 0-3 `(x, y, z, t)` -- what the network receives
- **Targets**: columns 4-7 `(Vx, Vy, P, TKE)` -- what the network must predict

**Task**: Split `loaded_data` into `inputs_np` and `targets_np` using NumPy slicing.

In [10]:
# YOUR CODE HERE
inputs_np = loaded_data[:, 0:4]   # shape (1000, 4)
targets_np = loaded_data[:, 4:8]  # shape (1000, 4)

In [11]:
# === TEST 8 ===
assert inputs_np.shape == (1000, 4), 'inputs shape: {}'.format(inputs_np.shape)
assert targets_np.shape == (1000, 4), 'targets shape: {}'.format(targets_np.shape)

# Verify correct columns
assert np.allclose(inputs_np[:, 0], loaded_data[:, 0]), 'First input col should be x'
assert np.allclose(inputs_np[:, 3], loaded_data[:, 3]), 'Last input col should be t'
assert np.allclose(targets_np[:, 0], loaded_data[:, 4]), 'First target col should be Vx'
assert np.allclose(targets_np[:, 3], loaded_data[:, 7]), 'Last target col should be TKE'

print('Test 8 passed!')
print('Inputs (coordinates): x, y, z, t')
print('  Ranges: ', end='')
for i, n in enumerate(['x', 'y', 'z', 't']):
    print('{}: [{:.2f}, {:.2f}]  '.format(n, inputs_np[:, i].min(), inputs_np[:, i].max()), end='')
print()
print('Targets (field vars): Vx, Vy, Pressure, TKE')
print('  Ranges: ', end='')
for i, n in enumerate(['Vx', 'Vy', 'P', 'TKE']):
    print('{}: [{:.1f}, {:.1f}]  '.format(n, targets_np[:, i].min(), targets_np[:, i].max()), end='')
print()

Test 8 passed!
Inputs (coordinates): x, y, z, t
  Ranges: x: [-0.49, 2.50]  y: [-0.50, 0.50]  z: [0.00, 0.00]  t: [0.00, 0.40]  
Targets (field vars): Vx, Vy, Pressure, TKE
  Ranges: Vx: [-1.0, 3.0]  Vy: [-1.0, 1.0]  P: [95000.3, 104977.5]  TKE: [0.3, 50.0]  


---
## Part C: Min-Max Normalization

**Why normalize?** Look at the ranges above. Pressure is ~100,000 while Vy is ~[-1, 1]. If you feed raw values to a neural network:
- Gradients will be dominated by Pressure (huge values)
- The network will barely learn Vy (tiny values)
- Training will be slow or diverge

**Min-max normalization** scales every feature to [0, 1], giving each equal importance.

---

### Exercise 9: Implement Min-Max Normalization (Per-Column)

**Formula**: `x_norm = (x - x_min) / (x_max - x_min)`

**Task**: Implement `minmax_normalize(data)` that:
1. Takes a 2D NumPy array of shape `(N, features)`
2. Computes `min`, `max`, and `range` per column (axis=0)
3. Handles the edge case where `range == 0` (constant column like z) by setting range to 1.0
4. Returns `(normalized_data, col_min, col_max, col_range)`

**Thesis connection**: This is exactly what `SpatioTemporalDataset.__init__` does. The `range == 0` guard is needed because your z column is constant (2D simulation slice).

In [ ]:
def minmax_normalize(data):
    """
    Min-max normalize a 2D array per column to [0, 1].
    
    Args:
        data: NumPy array of shape (N, features)
    Returns:
        (normalized_data, col_min, col_max, col_range)
        where col_min, col_max, col_range are 1D arrays of shape (features,)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# === TEST 9 ===
# Test on targets (has wide range: Pressure ~100k, TKE ~50)
normed_targets, t_min, t_max, t_range = minmax_normalize(targets_np)

assert normed_targets.shape == targets_np.shape, 'Shape should not change'
assert np.allclose(normed_targets.min(axis=0), 0.0, atol=1e-7), 'Min of each column should be ~0'
assert np.allclose(normed_targets.max(axis=0), 1.0, atol=1e-7), 'Max of each column should be ~1'
assert t_min.shape == (4,), 'col_min shape wrong'

# Test on inputs (has constant z column -> range should be set to 1)
normed_inputs, i_min, i_max, i_range = minmax_normalize(inputs_np)
z_col_idx = 2
assert i_range[z_col_idx] == 1.0, 'z column has range 0, should be set to 1.0 to avoid div-by-zero'
assert np.all(normed_inputs[:, z_col_idx] == 0.0), 'z column (all zeros) should normalize to all zeros'

# Check other columns are properly in [0, 1]
assert normed_inputs[:, 0].min() >= -1e-7 and normed_inputs[:, 0].max() <= 1.0 + 1e-7, 'x not in [0,1]'

print('Test 9 passed!')
print('\nBefore normalization (targets):')
for i, n in enumerate(['Vx', 'Vy', 'P', 'TKE']):
    print('  {}: [{:.1f}, {:.1f}], range={:.1f}'.format(n, t_min[i], t_max[i], t_range[i]))
print('After normalization: all columns in [0.0, 1.0]')

### Exercise 10: Implement Denormalization

After the network predicts normalized values in [0, 1], you must convert back to physical units for evaluation and visualization.

**Formula**: `x_original = x_norm * range + min`

**Task**: Implement `minmax_denormalize(normalized_data, col_min, col_range)` that reverses the normalization.

**Thesis connection**: Your `dataset.denormalize_targets()` does exactly this. It is called during visualization to show predictions in physical units.

In [ ]:
def minmax_denormalize(normalized_data, col_min, col_range):
    """
    Reverse min-max normalization.
    
    Args:
        normalized_data: array of shape (N, features) in [0, 1]
        col_min: array of shape (features,)
        col_range: array of shape (features,)
    Returns:
        Original-scale data
    """
    # YOUR CODE HERE
    pass

In [ ]:
# === TEST 10 ===
# Roundtrip: normalize then denormalize should give back original
recovered_targets = minmax_denormalize(normed_targets, t_min, t_range)
assert np.allclose(recovered_targets, targets_np, atol=1e-6), 'Roundtrip failed for targets'

recovered_inputs = minmax_denormalize(normed_inputs, i_min, i_range)
assert np.allclose(recovered_inputs, inputs_np, atol=1e-6), 'Roundtrip failed for inputs'

# Test with a known simple case
simple_data = np.array([[0.0, 100.0], [5.0, 200.0], [10.0, 300.0]])
s_norm, s_min, s_max, s_range = minmax_normalize(simple_data)
s_recovered = minmax_denormalize(s_norm, s_min, s_range)
assert np.allclose(s_recovered, simple_data), 'Simple roundtrip failed'

print('Test 10 passed!')
print('Normalization roundtrip verified: original -> normalize -> denormalize -> original')

### Exercise 11: Save Normalization Parameters as JSON

After training, you need the normalization parameters to reconstruct predictions at inference time. Your thesis saves them as JSON alongside the model `.pth` file.

**Task**: 
1. Create a dictionary `norm_params` with keys: `input_min`, `input_max`, `input_range`, `target_min`, `target_max`, `target_range` -- each as a Python list (not NumPy)
2. Save to JSON file
3. Load it back and verify

**Thesis connection**: Every trained model directory has a `*_normalization.json` file.

In [ ]:
import json

json_path = '../defense_prep/normalization_params.json'

# YOUR CODE HERE
# Step 1: Create the dictionary (use .tolist() to convert numpy arrays to lists)
norm_params = {
    'input_min': ...,
    'input_max': ...,
    'input_range': ...,
    'target_min': ...,
    'target_max': ...,
    'target_range': ...,
}

# Step 2: Save to JSON
# ...

# Step 3: Load back
loaded_params = ...

In [ ]:
# === TEST 11 ===
assert os.path.exists(json_path), 'JSON file not saved'

expected_keys = {'input_min', 'input_max', 'input_range', 'target_min', 'target_max', 'target_range'}
assert set(loaded_params.keys()) == expected_keys, 'Missing keys: {}'.format(expected_keys - set(loaded_params.keys()))

# Check values match
assert np.allclose(loaded_params['input_min'], i_min.tolist()), 'input_min mismatch'
assert np.allclose(loaded_params['target_range'], t_range.tolist()), 'target_range mismatch'

# Check all values are lists (JSON-serializable), not numpy arrays
for key, val in loaded_params.items():
    assert isinstance(val, list), '{} should be a list, got {}'.format(key, type(val))

print('Test 11 passed!')
print('Saved normalization params to:', json_path)
print('Keys:', list(loaded_params.keys()))

---
## Part D: PyTorch Dataset Class
---

### Exercise 12: Build a Custom Dataset Class

PyTorch's `Dataset` class defines how individual samples are accessed. A `DataLoader` then uses it to create batches.

**Task**: Create `MiniSpatioTemporalDataset(torch.utils.data.Dataset)` that:
1. `__init__(self, data_array)`: Takes the raw `(N, 8)` NumPy array, splits into inputs/targets, applies min-max normalization, stores normalization params, converts to float32 tensors
2. `__len__(self)`: Returns number of samples
3. `__getitem__(self, idx)`: Returns `(input_tensor, target_tensor)` for sample `idx`
4. `denormalize_targets(self, normalized)`: Reverses target normalization (accepts tensor or numpy)

**Thesis connection**: This matches your `SpatioTemporalDataset` in `unified_training_utils.py`.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MiniSpatioTemporalDataset(Dataset):
    def __init__(self, data_array):
        """
        Args:
            data_array: NumPy array of shape (N, 8) with columns [x,y,z,t,Vx,Vy,P,TKE]
        """
        # YOUR CODE HERE
        # 1. Split into inputs (cols 0-3) and targets (cols 4-7)
        # 2. Compute and store min, max, range per column for BOTH inputs and targets
        # 3. Handle range==0 case
        # 4. Normalize both to [0, 1]
        # 5. Convert to float32 torch tensors and store as self.inputs, self.targets
        pass
    
    def __len__(self):
        # YOUR CODE HERE
        pass
    
    def __getitem__(self, idx):
        # YOUR CODE HERE
        # Return (self.inputs[idx], self.targets[idx])
        pass
    
    def denormalize_targets(self, normalized):
        """
        Reverse normalization on target predictions.
        Accepts torch.Tensor or numpy array.
        Returns numpy array.
        """
        # YOUR CODE HERE
        pass

In [ ]:
# === TEST 12 ===
dataset = MiniSpatioTemporalDataset(synthetic_data)

# Check length
assert len(dataset) == 1000, 'Length should be 1000, got {}'.format(len(dataset))

# Check __getitem__
inp, tgt = dataset[0]
assert isinstance(inp, torch.Tensor), '__getitem__ should return tensors'
assert inp.shape == (4,), 'Input shape should be (4,), got {}'.format(inp.shape)
assert tgt.shape == (4,), 'Target shape should be (4,), got {}'.format(tgt.shape)
assert inp.dtype == torch.float32, 'Should be float32'

# Check normalization was applied
assert dataset.inputs.min() >= -1e-6, 'Inputs should be normalized to [0, 1]'
assert dataset.inputs.max() <= 1.0 + 1e-6, 'Inputs should be normalized to [0, 1]'
assert dataset.targets.min() >= -1e-6, 'Targets should be normalized to [0, 1]'
assert dataset.targets.max() <= 1.0 + 1e-6, 'Targets should be normalized to [0, 1]'

# Check denormalization roundtrip
denormed = dataset.denormalize_targets(dataset.targets)
assert isinstance(denormed, np.ndarray), 'denormalize_targets should return numpy'
assert np.allclose(denormed, targets_np, atol=1e-5), 'Denormalization roundtrip failed'

# Check denormalization works with torch tensors too
denormed_from_tensor = dataset.denormalize_targets(dataset.targets[:5])
assert isinstance(denormed_from_tensor, np.ndarray), 'Should return numpy even from tensor input'

print('Test 12 passed!')
print('Dataset: {} samples'.format(len(dataset)))
print('Input (normalized): [{:.3f}, {:.3f}]'.format(dataset.inputs.min().item(), dataset.inputs.max().item()))
print('Target (normalized): [{:.3f}, {:.3f}]'.format(dataset.targets.min().item(), dataset.targets.max().item()))

### Exercise 13: Create a DataLoader

A DataLoader wraps a Dataset and provides:
- **Batching**: Groups samples into mini-batches
- **Shuffling**: Randomizes order each epoch
- **Parallel loading**: `num_workers` for speed

**Task**: Create a DataLoader with `batch_size=64` and `shuffle=True`. Then iterate over one full epoch and count the batches.

**Thesis connection**: Your offline training uses `batch_size=512` with `shuffle=True`.

In [ ]:
BATCH_SIZE = 64

# YOUR CODE HERE
train_loader = ...

# Iterate one epoch and count batches
num_batches = 0
total_samples = 0
for batch_inputs, batch_targets in train_loader:
    num_batches += 1
    total_samples += batch_inputs.shape[0]

print('Batches per epoch:', num_batches)
print('Total samples processed:', total_samples)

In [ ]:
# === TEST 13 ===
import math
expected_batches = math.ceil(1000 / 64)
assert num_batches == expected_batches, 'Expected {} batches, got {}'.format(expected_batches, num_batches)
assert total_samples == 1000, 'Should process all 1000 samples, got {}'.format(total_samples)

# Check batch shapes
sample_inp, sample_tgt = next(iter(train_loader))
assert sample_inp.shape == (64, 4), 'Batch input shape should be (64, 4), got {}'.format(sample_inp.shape)
assert sample_tgt.shape == (64, 4), 'Batch target shape should be (64, 4), got {}'.format(sample_tgt.shape)

print('Test 13 passed!')
print('Batch shape: inputs={}, targets={}'.format(sample_inp.shape, sample_tgt.shape))

---
## Part E: Thinking Questions (Write Your Answers)

These questions will come up in your defense. Write short answers.

### Q1: Why min-max normalization instead of z-score (standardization)?

Z-score: `x_norm = (x - mean) / std` centers data at 0 with std=1.
Min-max: `x_norm = (x - min) / (max - min)` scales data to [0, 1].

Why did you choose min-max for your thesis? What could go wrong with z-score for this application?

**YOUR ANSWER:**

...

### Q2: Why save normalization params separately?

Why not just save them inside the model `.pth` file? Why a separate JSON file?

**YOUR ANSWER:**

...

### Q3: What happens if you normalize inputs and targets TOGETHER (all 8 columns at once) instead of separately?

Your code normalizes inputs `(x,y,z,t)` and targets `(Vx,Vy,P,TKE)` with separate min/max. What would go wrong if you did it as one `(N, 8)` normalization?

**YOUR ANSWER:**

...

### Q4: Your CSV has NO header. What would happen if you loaded it WITHOUT specifying column names?

Think about what pyarrow/pandas would do with the first row of data.

**YOUR ANSWER:**

...

### Q5: Compression context -- How big is your original dataset vs. your model?

Calculate:
- Original dataset size in bytes (7,919,100 rows x 8 columns x 4 bytes per float32)
- BaseCompressor model size (6,692 parameters x 4 bytes)
- What is the compression ratio?

Write the calculation below.

**YOUR ANSWER:**

...

---
## Summary: What You Covered Today

| Exercise | Concept | Direct Thesis Code Match |
|----------|---------|-------------------------|
| 1-5 | Tensor creation, slicing, reshape, numpy conversion, device | Used everywhere in pipeline |
| 6 | Understanding CSV data structure | `ML_test_loader_original_data.csv` |
| 7 | PyArrow CSV loading with manual column names | `SpatioTemporalDataset.__init__` |
| 8 | Splitting inputs from targets | `data[:, :4]` and `data[:, 4:]` |
| 9 | Min-max normalization with zero-range guard | Normalization block in Dataset |
| 10 | Denormalization | `dataset.denormalize_targets()` |
| 11 | Save normalization params as JSON | `*_normalization.json` files |
| 12 | Custom PyTorch Dataset class | `SpatioTemporalDataset` |
| 13 | DataLoader with batching | `DataLoader(batch_size=512)` |
| Q1-Q5 | Design decisions and thesis understanding | Defense questions |

**Tomorrow (Day 2)**: Linear layers, activation functions, loss functions, and building your first model from scratch.

---